# Vaches Noires experiments

This notebook is a mechanical conversion of the original Colab export used for the Vaches Noires experiments and the U-Net transfer-learning comparison. The unchanged export is preserved in `legacy/Vaches_Noires_original_colab_export.py`.

The non-public inputs and precomputed outputs are available in the [companion data folder](https://drive.google.com/drive/folders/1iC7QrdB2vZmaunjjPrIcu9r5wMeBPMlk).

Original notebook metadata:

Original file is located at
    https://colab.research.google.com/drive/1SUgTunaQ6yT3yXff9LjfajrBxFtQtVdz

# Comparaison Frangi-généralisé avec « U-net + Transfer learning »

Voir le Notebook python sur le papier « U-net + Transfer learning » : https://colab.research.google.com/drive/1fvgjQlK8pU4wUejU1DMLz1tJ4YL6w0tD?usp=sharing

# Première étape : Frangi généralisé :

## 1) Importation et sélection des données

In [ ]:
# mount google drive
import os

if 'drive' in os.listdir('./'):
    print('you have successfully mounted your google drive')
else:
    # mount google drive
    import os  # This line and the following lines should be indented
    if 'drive' in os.listdir('./'):
        print('you have successfully mounted your google drive')
    else:
        from google.colab import drive
        drive.mount('/content/drive')
        print('successfully mount your google drive, please restart the run time')
# copy files, new_label contains lots of images, so it could be a little slow.
# ! mkdir test
! cp -r ./drive/My\ Drive/crack_detection/test ./

dossier_drive = "./drive/MyDrive/crack_detection/"
chemin = "./test/"
dossier_test = "test/Resultats_UNet_ShiYongxiang/"
fichier = "Test1_512_512" # "VN_1"
dossier_VT = "test/"
dossier_image = "test/"

## Bibliothèques
import numpy as np
import imageio.v3 as iio
from skimage.color import rgb2gray
import matplotlib.pyplot as plt
!pip install hdbscan
import hdbscan
import pickle
##
## Bibliothèques
!pip install gudhi
from gudhi import SimplexTree
from scipy.sparse import coo_array
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse.csgraph import connected_components
from skimage.feature import hessian_matrix
from math import floor
##

Image = iio.imread(chemin + fichier + ".png")#[150:350,150:350]
# Convert RGBA to RGB by dropping the alpha channel
Image = Image[:,:,:3] # Select only the first 3 channels (RGB)
ImageNB = rgb2gray(Image)

plt.imshow(Image)
plt.colorbar()

n1,p1 = ImageNB.shape
π = np.pi
Infini =  2. #π/2
taille_police = 20
# n,p = ImageNB.shape
# N = n*p
PercolThreshold = 40
Σ_min = 10 # PercolThreshold // 2 # Pour le calcul des centroïdes
τ = 0.5 # Le seuil de la valeur moyenne de Frangi appliquée aux clusters
K = 2
expZ = 1
Σ = [1,3,5,7,9]
Rayon = 5 # 8
verbeux = True

fin_fichier = "K"+str(K)+"Rayon"+str(Rayon)+"expZ"+str(expZ)+"Σ"+str(Σ)+"PercolThreshold"+str(PercolThreshold)+".txt"

## 2) À partir de l'image en noir & blanc, on crée un (hyper-)graphe
(Sous forme de Simplex-Tree de la bibliothèque Gudhi)

In [ ]:
## Frangi

def Frangi(Im, σ=0.1, β = 1/2, type_image="creux") :
    A, C, B = hessian_matrix(Im, sigma=σ, order='rc', mode='mirror')# use_gaussian_derivatives=False) #Hrr, Hrc, Hcc
    A_plus_B = A + B
    A_moins_B = A - B
    Δ = np.sqrt(A_moins_B*A_moins_B + 4*C*C)

    Λ_un = 1/2 * (A_plus_B + Δ)
    Λ_deux = 1/2 * (A_plus_B - Δ)

    Λ_un_bis = np.where(np.abs(Λ_un) > np.abs(Λ_deux), Λ_deux, Λ_un)
    Λ_deux_bis = np.where(np.abs(Λ_un) > np.abs(Λ_deux), Λ_un, Λ_deux)

    Θ = (np.arccos(A_moins_B/Δ))/2

    Θ_bis = np.where(np.abs(Λ_un) > np.abs(Λ_deux), Θ + np.pi/2, Θ)
    Carré_norme_Hessienne = Λ_un**2 + Λ_deux**2 # Λ_deux_bis**2 # On prend la norme spectrale # Λ_un**2 + Λ_deux**2
    # The value of the threshold c depends on the grey-scale range of the image and half the value of the maximum Hessian norm has proven to work in most cases. However, future research will be directed towards automating this threshold selection.
    c = np.max(np.abs(Λ_deux_bis))# np.sqrt(np.max(Carré_norme_Hessienne))
    if type_image == "creux" :
        Fr = np.exp(-1/(2*β**2)*(Λ_un_bis/Λ_deux_bis)**2) * (1 - np.exp(-1/(2*((c/2)**2))*(Carré_norme_Hessienne))) * (Λ_deux_bis >= 0)
    else :
        Fr = np.exp(-1/(2*β**2)*(Λ_un_bis/Λ_deux_bis)**2) * (1 - np.exp(-1/(2*((c/2)**2))*(Carré_norme_Hessienne))) * (Λ_deux_bis <= 0)
    return Fr, (Λ_un_bis,Λ_deux_bis), Θ_bis, c

def Frangi_multiÉchelles(Im, Σ = [1,3,5,7,9], type_image="creux") :
    σ_max = max(Σ)
    n,p = Im.shape
    max_Frangi = np.zeros((n,p))
    liste_Frangi, liste_Λ, liste_Θ, liste_Norme_Hessienne  = [],[],[],[]
    for σ in Σ :
        print(σ)
        MatFr,(Λ_1,Λ_2), Θ, Norme_Hessienne = Frangi(Im, σ,type_image=type_image)
        liste_Frangi.append(MatFr[σ_max:-σ_max,σ_max:-σ_max])
        liste_Λ.append((Λ_1[σ_max:-σ_max,σ_max:-σ_max],Λ_2[σ_max:-σ_max,σ_max:-σ_max]))
        liste_Θ.append(Θ[σ_max:-σ_max,σ_max:-σ_max])
        liste_Norme_Hessienne.append(Norme_Hessienne)
        Condition = MatFr > max_Frangi
        max_Frangi = np.where(Condition, MatFr, max_Frangi)
    return max_Frangi[σ_max:-σ_max,σ_max:-σ_max], (liste_Frangi, liste_Λ, liste_Θ, liste_Norme_Hessienne)

def créer_Graphe_SimplexeTree(Im, Σ = [1,3,5,7,9], type_image="creux", verbeux=False) :
    # type_image : Veut-on repérer des "creux" (λ2 <= 0) ou des "bosses" (λ2 >= 0)
    # n,p = Im.shape
    # α = 1 # 1/2
    β = 1/2
    c = 1/4 # 1/2
    c_θ = 1/8 # 1/2

    Voisins = []
    for i in range(floor(Rayon+1)) :
        for j in range(floor(Rayon+1)) :
            if i**2 + j**2 <= Rayon**2 and (i,j) != (0,0) :
                Voisins.append((i,j))
    if verbeux :
        print("Liste de voisins possibles (Rayon=",Rayon,") : ", Voisins)
    mat_Frangi, (liste_Frangi, liste_Λ, liste_Θ, liste_Norme_Hessienne) = Frangi_multiÉchelles(Im, Σ)
    n,p = mat_Frangi.shape

    if verbeux :
        print("Matrice de Frangi calculée.")
        plt.figure("mat_Frangi")
        plt.title("Réponse de Frangi, σ ∈ "+str(Σ), fontsize=taille_police)
        plt.imshow(mat_Frangi)

    ST = SimplexTree()

    échelles_choisies = [0 for _ in range(len(Σ))]
    for i in range(n) :
        if verbeux and i%10 == 0 :
            print(i,n)
        for j in range(p) :
            index1 = i*p + j
            for voisin in Voisins :
                i2,j2 = i+voisin[0], j+voisin[1]
                if i2 < n and j2 < p :
                    distances = np.array([2*Infini for _ in Σ])#, dtype=float)
                    v = (np.array([i2,j2]) - np.array([i,j]))
                    norme_v = np.sqrt(v[0]**2 + v[1]**2)
                    v = v / norme_v
                    for σ in range(len(Σ)) :
                        r = 0
                        if norme_v <= 2*Σ[σ] : # 1/2 * # il faut que la distance ne soit pas trop grande devant l'échelle
                            θ, λ1, λ2, e_1, e_2, a,b = [0,0],[0,0],[0,0],[0,0],[0,0],[0,0],[0,0]
                            normeHessienne = liste_Norme_Hessienne[σ]
                            for index,indice in enumerate([(i,j),(i2,j2)]) :
                                θ[index] = liste_Θ[σ][indice]

                                λ1[index], λ2[index] = liste_Λ[σ][0][indice], liste_Λ[σ][1][indice]
                                e_1[index], e_2[index] = np.array([np.cos(θ[index]), np.sin(θ[index])]), np.array([-np.sin(θ[index]), np.cos(θ[index])])
                                coord1, coord2 =  v@e_1[index], v@e_2[index]
                                a[index],b[index] = coord1*np.sqrt(np.abs(λ1[index]/λ2[index])), coord2 #*np.sqrt(np.abs(λ2)/c)

                            if type_image == "creux" : # À CHANGER SELON LE TYPE D'IMAGE
                                if min(λ2) < 0 :
                                    r += Infini*2
                            else :
                                if max(λ2) > 0 :
                                    r += Infini*2
                            Similarité = (1-np.exp(-1/2 /(c**2) /(normeHessienne**2) *np.abs(λ2[0]*λ2[1])))
                            Similarité *= np.exp(-1/2 /(β**2) * (np.abs(λ1[0]/λ2[0])+np.abs(λ1[1]/λ2[1]))**2)
                            # # Similarité *= np.exp(-1/2 /(α**2) * (b[0]**2 + b[1]**2))
                            δθ = θ[0]-θ[1]
                            Similarité *= np.exp(-1/2 /(c_θ**2) * np.sin(δθ)**2)
                            '''δθ = np.abs(θ[0]-θ[1]) % π
                            if δθ < 0 :
                              δθ += π
                            δθ = min(δθ, π-δθ)
                            Similarité *= np.exp(-1/2 /(c_θ**2) * δθ)'''
                            # Similarité = np.exp(-1/2 /(α**2) * (a[0]**2 + b[0]**2 + a[1]**2 + b[1]**2))*(1-np.exp(-1/2 /(c**2) /(normeHessienne**2) *np.abs(λ2[0]*λ2[1])))
                            # r += np.sqrt(a**2 + b**2) * np.exp(-1/2*(1/2)**2 * (c/(normeHessienne**2)))# np.exp(1/2*(1/2)**2 * (normeHessienne**2/c)) # norme_v * #(normeHessienne/c)# * (c/normeHessienne) # À remplacer par les normes d'algèbre ?
                            distances[σ] =  1 - Similarité + r
                    σ0 = np.argmin(distances)
                    rayon = distances[σ0]
                    # rayon = min(distances)
                    index2 = i2*p + j2
                    if  0 < rayon < Infini/2 :
                        échelles_choisies[σ0] += 1
                    ST.insert([index1,index2], filtration=rayon)
    ST.prune_above_filtration(filtration=Infini/2)
    if verbeux :
        print("Simplexe Tree (graphe) calculé avec ", ST.num_vertices(), " nœuds.")
        print("Échelles choisies pour les distances : ", [échelle/sum(échelles_choisies) for échelle in échelles_choisies])

    return mat_Frangi, ST

## Receives a complex and returns the hierarchical clustering
# - ST is the complex coded by a Simplex Tree (Gudhi library)
# - K is the dimension of the K-polyhedra to look at
# - expZ is the exponent for the estimation of the density in HDBSCAN algorithm (for the excess of mass criterion). ƛ = 1/r^(expZ). HDBSCAN uses only expZ = 1 while expZ = p is more indicated for Euclidean data.
# - verbeux = verbose
# Returns: the labels of the points and their filtration (that is, the radius associated to the first K-simplex in which they appear
def SimplexeTree_vers_KPolyèdresHiérarchiques(ST, K=2, PercolThreshold=10, expZ=False, verbeux=False, n=None) :
    if n == None :
        n = ST.num_vertices()

    print(n)

    # For the (K-1)-Simplexes
    IndSimplexes_vers_Simplexes = []
    Simplexes_vers_IndSimplexes = {}

    DistancesSimplexes_row = []
    DistancesSimplexes_col = []
    DistancesSimplexes_data = []

    Filtrations_points = [[] for _ in range(n)]


    Simplexes = ST.get_skeleton(K)
    compteur_simplexes = 0
    nbSimplexes = 0
    if verbeux :
        print("Skeleton de dimension K=",K," calculé")

    for simplexe,r in Simplexes :
        if len(simplexe) == K+1 :
            compteur_simplexes += 1
            IndArêtes = []
            for i in simplexe :
                k_1_simplexe = simplexe.copy()
                k_1_simplexe.remove(i)
                k_1_simplexe = tuple(k_1_simplexe)
                if k_1_simplexe in Simplexes_vers_IndSimplexes :
                    indice = Simplexes_vers_IndSimplexes[k_1_simplexe]
                else :
                    indice = nbSimplexes
                    nbSimplexes += 1
                    Simplexes_vers_IndSimplexes[k_1_simplexe] = indice
                    IndSimplexes_vers_Simplexes.append(k_1_simplexe)
                IndArêtes.append(indice)

            # On ajoute les distances entre arêtes
            for IndArête1 in IndArêtes :
                for i in IndSimplexes_vers_Simplexes[IndArête1] :
                    Filtrations_points[i].append((r,IndArête1))

            IndArête1 = min(IndArêtes)
            IndArêtes.remove(IndArête1)
            for IndArête2 in IndArêtes :
                if IndArête2 > IndArête1 :
                    DistancesSimplexes_row.append(IndArête1)
                    DistancesSimplexes_col.append(IndArête2)
                    DistancesSimplexes_data.append(r)

    if verbeux :
        print("Complexe aux ",compteur_simplexes, "K-simplexes (arêtes du graphe) et aux ", nbSimplexes, "K-1 simplexes (servant de nœuds). Faisant un total de ", len(DistancesSimplexes_data), " arêtes (comptées avec multiplicité.")

    distancesSimplexes = coo_array((DistancesSimplexes_data, (DistancesSimplexes_row, DistancesSimplexes_col)), shape=(nbSimplexes, nbSimplexes)) #, dtype=float)
    if verbeux :
        print("Matrice des distances calculées.")

    nbCompo, labels = connected_components(distancesSimplexes)
    if verbeux :
        print(nbCompo, ' composantes connexes.')

    nombres = [0 for _ in range(nbCompo)]
    v_max,ind_max = -1,-1
    for i,l in enumerate(labels) :
        nombres[l] += 1
        if nombres[l] > v_max :
            v_max = nombres[l]
            ind_max = l
    if verbeux :
        print("Fait.")
    indices_points_classés = set()
    indices_simplexes = {}
    nouv_ind_simplexe = 0
    Liste_nouv_simplexes = []
    for i,l in enumerate(labels) :
        if l == ind_max :
            indices_simplexes[i] = nouv_ind_simplexe
            Liste_nouv_simplexes.append(i)
            nouv_ind_simplexe += 1
            for p in IndSimplexes_vers_Simplexes[i] :
                indices_points_classés.add(p)
    if verbeux :
        print("Seuls ", len(indices_points_classés), " points apparaissent dans la plus grande composante connexe pour ", len(Liste_nouv_simplexes), " simplexes.")

    for i in range(n) :
        if not i in indices_points_classés :
            Filtrations_points[i] = [(Infini, -1)]
        else :
            filtr_i = []
            for r,simplexe in Filtrations_points[i] :
                if simplexe in indices_simplexes :
                    filtr_i.append((r,simplexe))
            Filtrations_points[i] = filtr_i

    for i in range(n) :
        if len(Filtrations_points[i]) > 0 :
            liste_arêtes = sorted(Filtrations_points[i])
            Filtrations_points[i] = liste_arêtes[0]
        else :
            Filtrations_points[i] = (Infini,-1)
        # Sinon c'est que le points n'est pas classé

    Liste_nouv_ind_points = []
    nouv_ind = 0
    Dict_ancien_ind_points = {}

    for i in range(n) :
        if i in indices_points_classés :
            Liste_nouv_ind_points.append(i)
            Dict_ancien_ind_points[i] = nouv_ind
            nouv_ind += 1

    DistancesSimplexes_row2 = []
    DistancesSimplexes_col2 = []
    DistancesSimplexes_data2 = []
    for i,r in  enumerate(DistancesSimplexes_data) :
        if DistancesSimplexes_row[i] in indices_simplexes and DistancesSimplexes_col[i] in indices_simplexes :
            DistancesSimplexes_row2.append(indices_simplexes[DistancesSimplexes_row[i]])
            DistancesSimplexes_col2.append(indices_simplexes[DistancesSimplexes_col[i]])
            DistancesSimplexes_data2.append(r)

    if verbeux :
        print("Nouveau complexe (connexe) aux ",compteur_simplexes, "K-simplexes (arêtes du graphe) et aux ", len(indices_simplexes), "K-1 simplexes (servant de nœuds). Faisant un total de ", len(DistancesSimplexes_data2), " arêtes (comptées avec multiplicité.")

    distancesSimplexes2 = coo_array((DistancesSimplexes_data2, (DistancesSimplexes_row2, DistancesSimplexes_col2)), shape=(len(indices_simplexes),len(indices_simplexes)))#, dtype=float)
    if verbeux :
        print("Matrice des distances calculées.")

    nbCompo, labels = connected_components(distancesSimplexes2)
    if verbeux :
        print(nbCompo, ' composantes connexes.')
        print(labels.shape, labels)
        print(np.sum(labels == 0))
    Coord0 = distancesSimplexes2.coords[0].astype(np.int32)
    Coord1 = distancesSimplexes2.coords[1].astype(np.int32)
    distancesSimplexes2.coords = (Coord0, Coord1)
    print(distancesSimplexes2.coords)
    mst2 = minimum_spanning_tree(distancesSimplexes2) #, overwrite=True)

    if verbeux :
        print("minimum_spanning_tree")

    # Convert the graph to scipy.cluster.hierarchy array format
    mst2 = mst2.tocoo()

    mst_array = np.vstack([mst2.row, mst2.col, mst2.data]).T
    if verbeux :
        print(mst_array.shape)

    mst_array = mst_array[np.argsort(mst_array.T[2], kind="mergesort"), :]

    if verbeux :
        print("mst_array")

    # Convert edge list into standard hierarchical clustering format
    Z_Simplexes = _hierarchical_fast._single_linkage_label(mst_array)

    if verbeux :
        print("Z_Simplexes")

    nouv_IndSimplexes_vers_Simplexes =  [IndSimplexes_vers_Simplexes[simplexe] for simplexe in Liste_nouv_simplexes]
    for i,simplexe in enumerate(nouv_IndSimplexes_vers_Simplexes) :
        nouv_simpl = []
        for point in simplexe :
            nouv_simpl.append(Dict_ancien_ind_points[point])
        nouv_simpl.sort()
        nouv_simpl = tuple(nouv_simpl)
        nouv_IndSimplexes_vers_Simplexes[i] = nouv_simpl
    nouv_Filtrations_points = [Filtrations_points[ind] for ind in Liste_nouv_ind_points]
    for i,(r,ind_simplexe) in enumerate(nouv_Filtrations_points) :
        if ind_simplexe == -1 or not ind_simplexe in indices_simplexes :
            nouv_Filtrations_points[i] = (Infini,-1)
        else :
            nouv_ind_simplexe = indices_simplexes[ind_simplexe]
            nouv_Filtrations_points[i] = (r,nouv_ind_simplexe)

    Z = SimplexGraph_to_PointGraph(Z_Simplexes,  nouv_Filtrations_points, verbeux)

    if verbeux :
        print("Z", Z.shape)

    if expZ != False :
        Z = transform_Z(Z, f=lambda x : x**expZ)
        if verbeux :
            print('Transformation appliquée :  r -> r^',expZ)


    SLT = hdbscan.plots.SingleLinkageTree(Z)

    if verbeux :
        print("SLT")
        # plt.figure()
        # CT = hdbscan._hdbscan_tree.condense_tree(Z, min_cluster_size=(PercolThreshold-1))
        # Classe_CT = hdbscan.plots.CondensedTree(CT)
        # Classe_CT.plot(select_clusters=True, selection_palette=sns.color_palette())
        # plt.show()

    (labels, probabilities, stabilities, condensed_tree, single_linkage_tree) = hdbscan.hdbscan_._tree_to_labels(
        X=None,
        single_linkage_tree=Z,
        min_cluster_size=(PercolThreshold-1),
        cluster_selection_method="eom",
        allow_single_cluster=False,
        match_reference_implementation=False,
        cluster_selection_epsilon=0.0,
        max_cluster_size=0,
    ) # le premier paramètre X n'est pas utilisé...

    if verbeux :
        print("Labels (partiels) calculés.")

    labels_anciens_num = -np.ones((n,), dtype=int)
    for i,lab in enumerate(labels) :
        labels_anciens_num[Liste_nouv_ind_points[i]] = lab

    return labels_anciens_num, Filtrations_points


## Transform a hierarchical simplexes-clustering matrix to the hierarchical clustering matrix of the point cloud.
# The Z matrices have the scipy.cluster.hierarchy.linkage format
def SimplexGraph_to_PointGraph(Z_simplexes, Filtrations_points, verbose=False) :
    N = Z_simplexes.shape[0] +1 # le nombre de simplexes
    n = len(Filtrations_points)
    if verbose :
        print(N, n)
        print(Z_simplexes)
    Z = []
    Clusters_activés = {} # Un dictionnaire qui associe à : un numéro de cluster actif (non vide en points) -> le numéro de cluster dans Z_retour
    Tailles_clusters = {}

    # 2 types de fusion : des classiques et les simplexes sontenant plusieurs points

    Rayons = [(Z_simplexes[i,2], N+i) for i in range(N-1)] # (rayon, i in {0,N-2}) ou (rayon, (p1,p2))
    for point, (r, num_simplexe) in enumerate(Filtrations_points) :
        Rayons.append((r,(point, num_simplexe)))

    Rayons.sort(key=lambda x : (isinstance(x[1], int), x[0]))

    i = 0
    for r,c in Rayons :
        if isinstance(c, int) : # (Z_simplexes[2], i)
            cluster1, cluster2 = round(Z_simplexes[c-N,0]), round(Z_simplexes[c-N,1])
            if (cluster1 in Clusters_activés) and (cluster2 in Clusters_activés) :
                nouv_taille = Tailles_clusters[cluster1] + Tailles_clusters[cluster2]
                Z.append([Clusters_activés[cluster1], Clusters_activés[cluster2], r, nouv_taille])
                Clusters_activés[c] = n + i
                Tailles_clusters[c] = nouv_taille
                i += 1
            elif cluster1 in Clusters_activés :
                Clusters_activés[c] = Clusters_activés[cluster1]
                Tailles_clusters[c] = Tailles_clusters[cluster1]
            elif cluster2 in Clusters_activés :
                Clusters_activés[c] = Clusters_activés[cluster2]
                Tailles_clusters[c] = Tailles_clusters[cluster2]
        else :
            p,num_simplexe = c
            if num_simplexe in Clusters_activés :
                nouv_taille = Tailles_clusters[num_simplexe] +1
                Z.append([p, Clusters_activés[num_simplexe], r, nouv_taille])
                Clusters_activés[num_simplexe] = n + i
                Tailles_clusters[num_simplexe] = nouv_taille
                i += 1
            else :
                Clusters_activés[num_simplexe] = p
                Tailles_clusters[num_simplexe] = 1

    Z = np.array(Z) #, dtype=float)
    if verbose :
        print(len(Clusters_activés),Z.shape)
        print(Z)

    return Z

## Apply a transformation to the third column of the Z-linkage (scipy.cluster.hierarchy) matrix ; the column associated to the RADIUS (expZ parameter)
def transform_Z(Z, f=lambda x : x**2) :
    N = len(Z)
    Z2 = np.copy(Z)
    for i in range(N) :
        Z2[i,2] = f(Z[i,2])
    return Z2

MatFrangi, ST = créer_Graphe_SimplexeTree(ImageNB, Σ = Σ, type_image="creux", verbeux=verbeux)
n,p = MatFrangi.shape
N = n * p

plt.imshow(MatFrangi)
plt.colorbar()

ST.expansion(K)

print("Calcul du simplex tree étendu à la dimension K = ", K, "\n Il a maintenant ", ST.num_vertices(), " nœuds pour ", ST.num_simplices(), " simplexes.")

## bibliothèques

from sklearn.cluster import _hierarchical_fast
##

labels, Filtrations = SimplexeTree_vers_KPolyèdresHiérarchiques(ST=ST, K=K, PercolThreshold=PercolThreshold, expZ=expZ, verbeux=verbeux, n=N)

print("Les filtrations sur les points sont maintenant calculées.")

def Calcul_forêt(ST, Filtrations_points) :
    Arêtes = ST.get_skeleton(1)
    DistancesPoints_row = []
    DistancesPoints_col = []
    DistancesPoints_data = []

    for simplexe,r in Arêtes :
        if len(simplexe) == 2 :
            i,j = simplexe
            x1,y1 = i//p, i%p
            x2,y2 = j//p,j%p
            dist = np.sqrt((x1-x2)**2 + (y1-y2)**2)
            filtration = max([Filtrations_points[i][0],Filtrations_points[j][0],r*dist])
            if filtration < Infini/2 :
                DistancesPoints_row.append(i)
                DistancesPoints_col.append(j)
                DistancesPoints_data.append(filtration) # on prend un graphe non pondéré ? ou bien pondéré par la similarité (1 - r)/distance

    Graphe_coo = coo_array((DistancesPoints_data, (DistancesPoints_row, DistancesPoints_col)), shape=(N,N), dtype=float)
    Coord0 = Graphe_coo.coords[0].astype(np.int32)
    Coord1 = Graphe_coo.coords[1].astype(np.int32)
    Graphe_coo.coords = (Coord0, Coord1)
    Forêt = minimum_spanning_tree(Graphe_coo)#, overwrite=True) # CSR matrix Compressed Sparse Row

    nbCompo, labels2 = connected_components(Forêt)
    if verbeux :
        print(nbCompo, ' composantes connexes dans le graphe.')
    nombres = [0 for _ in range(nbCompo)]
    v_max,ind_max = -1,-1
    # valeurs, nombres = np.unique(labels, return_counts=True)
    for i,l in enumerate(labels2) :
        nombres[l] += 1
        if nombres[l] > v_max :
            v_max = nombres[l]
            ind_max = l
    # v_max = np.argmax(nombres)
    Nœuds = set()
    for i,l in enumerate(labels2) :
        if l == ind_max :
            Nœuds.add(i)

    Forêt = Forêt + Forêt.T

    if verbeux :
        print("La composante principale du graphe comporte ", len(Nœuds), " nœuds.")
    return Forêt, Graphe_coo, Nœuds

Forêt, Graphe_coo, Nœuds = Calcul_forêt(ST=ST, Filtrations_points=Filtrations)

Filtrations_points = [r for r,_ in Filtrations]

nom_fichier = dossier_drive + "ArbresHierarchiques/" + fichier+"K"+str(K)+"Rayon"+str(Rayon)+"expZ"+str(expZ)+"Σ"+str(Σ)+"PercolThreshold"+str(PercolThreshold)+".txt"
# print(os.listdir('./'), os.listdir('./drive/'))
with open(nom_fichier, 'wb') as f :
    pickle.dump((Graphe_coo, Forêt, Nœuds, Filtrations_points), f )
    print("Enregistrement de la forêt sur les points.'")

Filtrations_points = np.array(Filtrations_points).reshape((n,p))
plt.figure()
plt.title("filtration")
plt.imshow(np.where(1-Filtrations_points > 0, 1-Filtrations_points, 0))
plt.colorbar()

# Extraction des centroïdes

In [ ]:
def DFS_feuilles(A, i, Nœuds) : # parcourt en longueur du graphe à partir du nœud i. Renvoie tous les couples (feuilles, d(i,feuille))
  Q = [(i,0)]
  Explorés = {i}
  retour = []
  while len(Q) > 0 :
    v,Σ = Q.pop()
    Voisins = [f for f in A[[v]].indices if f in Nœuds]
    Poids = [A[v,f] for f in Voisins]
    feuille = True
    for ind,voisin in enumerate(Voisins) :
      ω = Poids[ind]
      if not voisin in Explorés :
        feuille = False
        Explorés.add(voisin)
        Q.append((voisin, Σ + ω))
    if feuille :
        retour.append((v, Σ))
  return retour

def DFS_chemin(A, départ, cible, Nœuds) : # parcourt en longueur du graphe à partir du nœud départ pour identifier le chemin qui mène à cible.
  Q = [(départ,[départ])]
  Explorés = {départ}
  retour = []
  while len(Q) > 0 :
    v,chemin = Q.pop()
    if v == cible :
      return chemin
    Voisins, Poids = A[[v]].indices, A[[v]].data
    for ind,voisin in enumerate(Voisins) :
      # ω = Poids[ind]
      if not voisin in Explorés and voisin in Nœuds :
        Explorés.add(voisin)
        Q.append((voisin, chemin + [voisin]))#,  Σ + ω))^

def DFS_sous_branches(A, i, Nœuds) : # Renvoie une liste de tous les sous-arbres si l'on coupe au niveau du nœud i
  Fils_i = [f for f in A[[i]].indices if f in Nœuds]
  nb_fils = len(Fils_i)
  Q = [[f] for f in Fils_i]
  Nœuds_sous_branches = [{i} for _ in range(nb_fils)]

  for a,f in enumerate(Fils_i) :
    Nœuds_sous_branches[a].add(f)
    while len(Q[a]) > 0 :
      v = Q[a].pop()
      Voisins, Poids = A[[v]].indices, A[[v]].data
      for ind,voisin in enumerate(Voisins) :
        if not voisin in Nœuds_sous_branches[a] and voisin in Nœuds :
          Nœuds_sous_branches[a].add(voisin)
          Q[a].append(voisin)
    # Nœuds_sous_branches[a] = Explorés #- {i}
  return Nœuds_sous_branches


def milieu_chemin(A, chemin, Σ) : # renvoie le nœud au milieu du chemin
  Somme = 0
  i = 1
  while i < len(chemin) :
    Somme += A[chemin[i-1], chemin[i]]
    if Somme > Σ/2 :
      return chemin[i-1]
    i += 1


def Centroïdes(A, K, Nœuds, Σ_min=0) :
  if K == 0 :
    return set()
  else :
    p = min(Nœuds)
    # Fils_i = [f for f in A[[p]].indices if f in Nœuds]
    Distances_p = DFS_feuilles(A, p, Nœuds)
    # print(Distances_p)
    p1,_ = max(Distances_p, key=lambda x: x[1])
    Distances_p1 = DFS_feuilles(A, p1, Nœuds)
    p2,Σ =  max(Distances_p1, key=lambda x: x[1])
    if Σ > Σ_min :
      chemin = DFS_chemin(A, p1, p2, Nœuds)
      # print(chemin,p1,p2, Σ)
      centroïde = milieu_chemin(A,chemin,Σ)
      print(centroïde, Σ)
      # print(centroïde)
      Nœuds_sous_branches = DFS_sous_branches(A,centroïde,Nœuds)
      nb_fils = len(Nœuds_sous_branches)
      # if nb_fils != 3 :
      #   print("nb_fils=",nb_fils)
      Centres = {centroïde}
      for a in range(nb_fils) :
        # print((K-1)//nb_fils, Nœuds_sous_branches[a])
        centres_fils = Centroïdes(A,(K-1)//nb_fils, Nœuds_sous_branches[a], Σ_min)
        Centres = Centres | centres_fils
      return Centres
    else :
      print("Petite branche non prise en compte : Σ <= Σ_min ", Σ, " <= ", Σ_min)
      return set()

nom_fichier = dossier_drive + "ArbresHierarchiques/" + fichier+ fin_fichier

with open(nom_fichier, 'rb') as f:
    (Graphe_coo, Forêt, Nœuds, Filtrations_points) = pickle.load(f)
    print("Chargement ")

## Transformation filtration -> poids

N,_ = Forêt.shape
n = round(np.sqrt(N))
p = n


δn = (n1-n)//2
δp = (p1-p)//2
print("Dimensions : ",(n,p), " décalage : ", (δn,δp))

for i in range(N) :
    Voisins = Forêt[[i]].indices
    for j in Voisins :
        if j > i :
            # i1,j1 = i//p + δn, i%p + δp
            # x1,y1 = j1, i1
            # i2,j2 = j//p + δn, j%p + δp
            # x2,y2 = j2, i2
            # rayon = np.sqrt((x1-x2)**2 + (y1-y2)**2)
            filtr = Forêt[i,j] #/ rayon
            if filtr > 1 :
                r = 0
            else :
                r = 1 - filtr
            # r = np.exp(-1/2 * (filtr/τ)**2) #* rayon
            Forêt[i,j] = r
            Forêt[j,i] = r

print("Transformation filtration -> poids opérée.")

##

Nb_Centroïdes = 100000

Centres = Centroïdes(Forêt, Nb_Centroïdes, Nœuds, Σ_min = Σ_min)
print(len(Centres), " centres ont été identifiés. ", Nb_Centroïdes, " centres demandés. Il apparaissent sur le dessin ci-dessous en plus gros et rouge.", Centres)

nom_fichier = dossier_drive + "ArbresHierarchiques/" + fichier+ "CentroidesΣ_min"+str(Σ_min) + "τ"+str(τ)+ fin_fichier

with open(nom_fichier, 'wb') as f:
    pickle.dump((Graphe_coo, Forêt, Nœuds, Filtrations_points, Centres), f)
    print("Enregistrement de la forêt et des centres sur les points.'")

# Extraction hiérarchiques des fractures

In [ ]:
import networkx as nx

fin_fichier = "K"+str(K)+"Rayon"+str(Rayon)+"expZ"+str(expZ)+"Σ"+str(Σ)+"PercolThreshold"+str(PercolThreshold)+".txt"
nom_fichier = dossier_drive + "ArbresHierarchiques/" + fichier+ "CentroidesΣ_min"+str(Σ_min) + "τ"+str(τ)+ fin_fichier

with open(nom_fichier, 'rb') as f:
    (Graphe_coo, Forêt, Nœuds, Filtrations_points, Centres) = pickle.load(f)
    print("Chargement ")


N,_ = Forêt.shape
n = round(np.sqrt(N))
p = n


# Graphe_csr = Graphe_coo.tocsr()
#
# print("Conversion en csr.")
#
#
δn = (n1-n)//2
δp = (p1-p)//2
print("Dimensions : ",(n,p), " décalage : ", (δn,δp))
#
# # Graphe_csr = -1/2 * (Graphe_csr/τ)**2
# #
# # print("Première transformation.")
# #
# # Graphe_csr = Graphe_csr.expm1()
#
# x,y = Graphe_csr.nonzero()
#
# print(len(x))
#
# for a in range(len(x)) :
#     i,j = x[a], y[a]
#     # i1,j1 = i//p + δn, i%p + δp
#     # x1,y1 = j1, i1
#     # i2,j2 = j//p + δn, j%p + δp
#     # x2,y2 = j2, i2
#     # rayon = np.sqrt((x1-x2)**2 + (y1-y2)**2)
#     # # if j > i :
#     filtr = Graphe_csr[i,j]#/rayon
#     r = np.exp(-1/2 * (filtr/τ)**2)#*rayon
#     Graphe_csr[i,j] = r
#     # Graphe_csr[j,i] = r
#
#
# print("Transformation filtration -> poids opérée.")



Graphe = nx.from_scipy_sparse_array(Graphe_coo)

print("Graphe au format NetworkX.")

# Création

Distances_centres_row = []
Distances_centres_col = []
Distances_centres_data = []

for i,centre in enumerate(Centres) :
    print("Centre n° ",i,"/",len(Centres))
    distances = nx.single_source_dijkstra_path_length(Graphe, centre, cutoff=4*Σ_min, weight='weight')
    # print("Djikstra.")
    for c2 in Centres :
        if c2 in distances and centre < c2 :
            # Distances_centres[centre,c2] = distances[c2]
            # Distances_centres[c2,centre] = distances[c2]
            Distances_centres_row.append(centre)
            Distances_centres_col.append(c2)
            Distances_centres_data.append(distances[c2])


Distances_centres = coo_array((Distances_centres_data, (Distances_centres_row, Distances_centres_col)), shape=(N,N), dtype=float)

nom_fichier = dossier_drive + "ArbresHierarchiques/" + fichier+ "Distances_centresΣ_min"+str(Σ_min) + "τ"+str(τ)+ fin_fichier

with open(nom_fichier, 'wb') as f:
    pickle.dump(Distances_centres, f)
    print("Enregistrement de Distances_centres.'")

## Chargement au lieu de création

nom_fichier = dossier_drive + "ArbresHierarchiques/" + fichier+ "Distances_centresΣ_min"+str(Σ_min) + "τ"+str(τ)+ fin_fichier

with open(nom_fichier, 'rb') as f:
    Distances_centres = pickle.load(f)
    print("Chargement distances_centres")

# Arbre sur les centroïdes

In [ ]:
Graphes_centres = nx.from_scipy_sparse_array(Distances_centres)
print("Graphe des distances entre centres calculé.")

Arbre_centres = nx.minimum_spanning_tree(Graphes_centres)

print("Ainsi que l'arbre.")

Filtration_Arêtes_Arbre = []

for i,(c1,c2) in enumerate(Arbre_centres.edges) :
    print(i)
    distance, path = nx.single_source_dijkstra(Graphe, c1, target=c2)#, cutoff=Σ_min, weight='weight')
    filtr = distance / (len(path)-1)
    if filtr > 1 :
        filtr = 0
    else :
        filtr = 1 - filtr
    # filtr = np.exp(-1/2 * (filtr/τ)**2)

    n_préc = path[0]
    for n_suiv in path[1:] :
        Filtration_Arêtes_Arbre.append(((n_préc,n_suiv),filtr))
        n_préc = n_suiv



Filtration_Arêtes_Arbre.sort(key=lambda x : x[1], reverse=True)


# Filtration_Nœuds_Arbre = [(nœud, Filtrations_points[nœud]) for nœud in Nœuds_Arbre]

# Filtration_Nœuds_Arbre.sort(key=lambda x : x[1])

print("Les nœuds de de l'arbre couvrant les centres ont été identifiés.")


nom_fichier = dossier_drive + "ArbresHierarchiques/" + fichier+ "Filtration_Nœuds_ArbreΣ_min"+str(Σ_min) + "τ"+str(τ)+ fin_fichier


with open(nom_fichier, 'wb') as f:
    pickle.dump((Graphe_coo, Forêt, Nœuds, Filtrations_points, Centres, Arbre_centres, Filtration_Arêtes_Arbre), f)
    print("Enregistrement de la forêt et des centres sur les points et de Filtration_Arêtes_Arbre.'")

# Dessin des fractures

In [ ]:
nom_fichier = dossier_drive + "ArbresHierarchiques/" + fichier+ "Filtration_Nœuds_ArbreΣ_min"+str(Σ_min) + "τ"+str(τ)+ fin_fichier


with open(nom_fichier, 'rb') as f:
    (Graphe_coo, Forêt, Nœuds, Filtrations_points, Centres, Arbre_centres, Filtration_Arêtes_Arbre) = pickle.load(f)
    print("Ouverture de la forêt et des centres sur les points et de Filtration_Arêtes_Arbre.'")


grandissement = 1 # 3






N,_ = Forêt.shape
n = round(np.sqrt(N))
p = n
n_image,_ = ImageNB.shape

VT_Image = iio.imread(dossier_drive + dossier_VT + "VT_" + fichier + ".png")[:,:,1:]
VT_ImageNB = rgb2gray(VT_Image)
n_VT,_ = VT_ImageNB.shape
if n_VT != n_image :
    from skimage.transform import resize # rescale
    VT_ImageNB = resize(VT_ImageNB, (n_image,n_image)) # rescale(VT_ImageNB, n_image/n_VT, anti_aliasing=True)


δn = (n_image-n)//2
δp = δn

résultats_UNet = np.load(dossier_drive + dossier_test + fichier + "_label.npy")
résultats_UNet = résultats_UNet.astype(float) # Conversion Bool => Float
print(résultats_UNet.shape)

VT_ImageNB = VT_ImageNB[δn:-δn,δn:-δn]
résultats_UNet = résultats_UNet[δn:-δn,δn:-δn]

if not(résultats_UNet.shape == VT_ImageNB.shape == (n,n)) :
    print("not(résultats_UNet.shape == VT_ImageNB_grossie.shape == n)", résultats_UNet.shape, VT_ImageNB.shape, n, n_image)
    '''mn,mp = VT_ImageNB_grossie.shape
    if mp != n :
        m = mp
    else :
        m = mn
    VT_ImageNB_grossie = VT_ImageNB_grossie.astype(float)
    VT_ImageNB_grossie = rescale(VT_ImageNB_grossie, n/m, anti_aliasing=True)
    if not(résultats_UNet.shape == VT_ImageNB_grossie.shape == (n,n)) :
        print(résultats_UNet.shape, VT_ImageNB_grossie.shape, n)'''
    1/0

plt.figure(figsize=[6*grandissement, 6*grandissement]) #[6.4, 4.8]
plt.imshow(Image[δn:-δn,δn:-δn], alpha=0.65) # 0.35


Alpha = (résultats_UNet >= 1).astype(float).T

X,Y = np.nonzero(Alpha)

plt.scatter(X,Y,alpha=0.9,s=0.4**2,color=[189/255,89/255,22/255])

# ax.imshow(Alpha, alpha=Alpha, **imshow_kwargs) # 0.35
# plt.colorbar()

# plt.imshow(Image[δn:-δn,δn:-δn], alpha=0.65) # 0.35

plt.show()



plt.figure(figsize=[6*grandissement, 6*grandissement]) #[6.4, 4.8]


plt.imshow(Image, alpha=0.65) # 0.35

# plt.plot([0,100], [100,300])

décompte = 0

Centres_dessinés = set()

for (nœud1,nœud2),filtr in Filtration_Arêtes_Arbre :
# for nœud1 in range(N) :
#     for nœud2 in Forêt[[nœud1]].indices :
#         if nœud2 > nœud1 :
#             filtr = Forêt[nœud1,nœud2]
    i1,j1 = nœud1//p + δn, nœud1%p + δp
    x1,y1 = j1, i1
    i2,j2 = nœud2//p + δn, nœud2%p + δp
    x2,y2 = j2, i2
    plt.plot([x1,x2],[y1,y2],alpha=0.7,linewidth=1.0,color=[189/255,89/255,22/255])
    if nœud1 in Centres and not nœud1 in Centres_dessinés :
        # plt.scatter(x1,y1, c='k', marker='*',alpha=0.10)
        Centres_dessinés.add(nœud1)
    if nœud2 in Centres and not nœud2 in Centres_dessinés :
        # plt.scatter(x2,y2, c='k', marker='*',alpha=0.10)
        Centres_dessinés.add(nœud2)

    décompte += 1

    if filtr < 0.30 : # 0.174762 : # Tversky(1;0,5) # 0.184931 : # Wassterstein #0.2786082632313 : Jaccard # np.exp(-1/2 * (1.5/τ)**2) :
        break # ()

# Différents tests avec la comparaison de l'U-Net + Transfer learning https://colab.research.google.com/drive/1fvgjQlK8pU4wUejU1DMLz1tJ4YL6w0tD?usp=sharing

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy.ndimage import binary_dilation

def rgba_mask(mask, color_rgba):
    out = np.zeros((*mask.shape, 4), dtype=float)
    out[mask.astype(bool)] = color_rgba
    return out

def _thicken(mask, r):
    if r <= 0:
        return mask.astype(bool)
    struct = np.ones((2*r + 1, 2*r + 1), dtype=bool)
    return binary_dilation(mask.astype(bool), structure=struct)

def plot_fault_overlays(ImageNB, VT_ImageNB, A, resultats_UNet,
                        alpha_img=0.7,
                        thicken_px=2,
                        use_contours=True,
                        contour_width=2.5,
                        line_alpha=0.5,     # transparence des contours (A & U-Net)
                        mask_alpha=0.4,     # transparence du masque (GT)
                        legend_fontsize=17,
                        savepath=None, dpi=200):

    bleu_foncé = (0.00, 0.20, 0.80, mask_alpha)   # #0033CC
    bleu_lisible = (0.12, 0.47, 0.71, mask_alpha) # #1F77B4
    bleu_nuit = (0.00, 0.20, 0.40, mask_alpha)
    bleu_pétrole = (0.00, 0.30, 0.60, mask_alpha)
    # Couleurs (ordre voulu : GT, U-Net, Frangi)
    colors = {
        "Vérité terrain":            (1.0, 0.0, 0.0, mask_alpha),  # rouge
        "U-Net + transfer learning": (0.1, 1.0, 0.1, mask_alpha),  # vert
        "FRANGI-généralisé":         bleu_foncé # (0.0, 0.6, 1.0, mask_alpha),  # bleu clair
    }
    colors_rgb = {k: c[:3] for k, c in colors.items()}

    # Épaississement
    VT = _thicken(VT_ImageNB, thicken_px)
    UN = _thicken(resultats_UNet, thicken_px)
    FR = _thicken(A, thicken_px)

    fig, ax = plt.subplots(figsize=(10, 10), constrained_layout=True, dpi=dpi)
    ax.imshow(ImageNB, cmap="gray", alpha=alpha_img)

    if use_contours:
        # GT en masque plein
        ax.imshow(rgba_mask(VT, colors["Vérité terrain"]))
        # D'abord U-Net, puis Frangi en contours
        ax.contour(UN, levels=[0.5], colors=[colors_rgb["U-Net + transfer learning"]],
                   linewidths=contour_width, alpha=line_alpha)
        ax.contour(FR, levels=[0.5], colors=[colors_rgb["FRANGI-généralisé"]],
                   linewidths=contour_width, alpha=line_alpha)

        legend_handles = [
            Patch(color=colors["Vérité terrain"], label="Vérité terrain"),
            Line2D([0], [0], color=colors_rgb["U-Net + transfer learning"], lw=contour_width,
                   alpha=line_alpha, label="U-Net + transfer learning"),
            Line2D([0], [0], color=colors_rgb["FRANGI-généralisé"], lw=contour_width,
                   alpha=line_alpha, label="FRANGI-généralisé"),
        ]
    else:
        # Tous en masques pleins : GT, puis U-Net, puis Frangi
        ax.imshow(rgba_mask(VT, colors["Vérité terrain"]))
        ax.imshow(rgba_mask(UN, colors["U-Net + transfer learning"]))
        ax.imshow(rgba_mask(FR, colors["FRANGI-généralisé"]))

        legend_handles = [
            Patch(color=colors["Vérité terrain"],            label="Vérité terrain"),
            Patch(color=colors["U-Net + transfer learning"], label="U-Net + transfer learning"),
            Patch(color=colors["FRANGI-généralisé"],         label="FRANGI-généralisé"),
        ]

    ax.legend(handles=legend_handles, loc="lower right", frameon=True,
              fontsize=legend_fontsize)
    ax.set_axis_off()

    if savepath is not None:
        plt.savefig(savepath, bbox_inches="tight", dpi=dpi)
    plt.show()


def dessine_segment(A,x1,y1,x2,y2) :
    if x1 == x2 :
        for y in range(y1,y2+1) :
            A[x1,y] += 1
    else :
        if x2 < x1 :
            a,b = x1,x2
            x1,x2 = b,a
            a,b = y1,y2
            y1,y2 = b,a
        pente = (y2 - y1)/(x2-x1)
        for x in range(x1,x2+1) :
            y = round(y1 + pente*(x-x1))
            A[x,y] += 1

##
!pip install pot
import ot # Python Optimal Transport
##

Fichiers = ["Test1_512_512"] # ["Test1_512_512", "Test2_512_512"]

recherche_meilleure_filtration = True
α, β = 1, 1/2 # pour l'indice de Tversky
rayon_Jaccard = 6
from scipy.signal import convolve2d
Kernel = np.zeros((2*rayon_Jaccard+1,2*rayon_Jaccard+1))

for i in range(2*rayon_Jaccard+1) :
    for j in range(2*rayon_Jaccard+1) :
        if (i-rayon_Jaccard)**2 + (j-rayon_Jaccard)**2 <= rayon_Jaccard**2 :
            Kernel[i,j] = 1


filtration_choisie = 0.3
Mesures = ["Tversky", "Jaccard", "Wasserstein"]


filtrations_à_tester = np.arange(0.301,0.10,-1/500)



fin_fichier = "K"+str(K)+"Rayon"+str(Rayon)+"expZ"+str(expZ)+"Σ"+str(Σ)+"PercolThreshold"+str(PercolThreshold)+".txt"


for fichier in Fichiers :
    Meilleures_filtrations = {"Tversky":[0,np.inf], "Jaccard":[0,np.inf], "Wasserstein":[np.inf,np.inf]} # couple (valeur_filtration, valeur_mesure)
    déjà_fait = False
    print(fichier)
    Image = iio.imread(dossier_drive + dossier_image + fichier + ".png")
    # Convert RGBA to RGB by dropping the alpha channel
    Image = Image[:,:,:3] # Select only the first 3 channels (RGB)
    ImageNB = rgb2gray(Image)
    n_image,_ = ImageNB.shape

    VT_Image = iio.imread(dossier_drive + dossier_VT + "VT_" + fichier + ".png")[:,:,1:]
    VT_ImageNB = rgb2gray(VT_Image)
    n_VT,_ = VT_ImageNB.shape
    if n_VT != n_image :
        from skimage.transform import resize # rescale
        VT_ImageNB = resize(VT_ImageNB, (n_image,n_image)) # rescale(VT_ImageNB, n_image/n_VT, anti_aliasing=True)
    VT_ImageNB = VT_ImageNB > VT_ImageNB.max()/2
    # VT_ImageNB = VT_ImageNB.T

    VT_ImageNB_grossie = convolve2d(VT_ImageNB, Kernel, mode="same", boundary="symm")
    VT_ImageNB_grossie = VT_ImageNB_grossie >= 1

    Index_VT = np.argwhere(VT_ImageNB > 0)
    ns,_ = Index_VT.shape

    nom_fichier = dossier_drive + "ArbresHierarchiques/" + fichier+ "Filtration_Nœuds_ArbreΣ_min"+str(Σ_min) + "τ"+str(τ)+ fin_fichier
    with open(nom_fichier, 'rb') as f:
        Graphe_coo, _, _, _, _, _, Filtration_Arêtes_Arbre = pickle.load(f)
        print("Lecture Filtration_Arêtes_Arbre de "+fichier)
    N,_ = Graphe_coo.shape
    n = round(np.sqrt(N))

    δn = (n_image-n)//2

    résultats_UNet = np.load(dossier_drive + dossier_test + fichier + "_label.npy")
    résultats_UNet = résultats_UNet.astype(float) # Conversion Bool => Float

    VT_ImageNB_grossie = VT_ImageNB_grossie[δn:-δn,δn:-δn]
    résultats_UNet = résultats_UNet[δn:-δn,δn:-δn]

    if not(résultats_UNet.shape == VT_ImageNB_grossie.shape == (n,n)) :
        print("not(résultats_UNet.shape == VT_ImageNB_grossie.shape == n)", résultats_UNet.shape, VT_ImageNB_grossie.shape, n)
        '''mn,mp = VT_ImageNB_grossie.shape
        if mp != n :
            m = mp
        else :
            m = mn
        VT_ImageNB_grossie = VT_ImageNB_grossie.astype(float)
        VT_ImageNB_grossie = rescale(VT_ImageNB_grossie, n/m, anti_aliasing=True)
        if not(résultats_UNet.shape == VT_ImageNB_grossie.shape == (n,n)) :
            print(résultats_UNet.shape, VT_ImageNB_grossie.shape, n)'''
        1/0
    Rayons = filtrations_à_tester.copy()
    index_rayon = 0
    rayon_suivant = Rayons[index_rayon]

    A = résultats_UNet.copy()#.T

    Index_A = np.argwhere(A >= 1)
    nt,_ = Index_A.shape

    A_bin = A >= 1
    A_Jaccard = convolve2d(A_bin, Kernel, mode="same", boundary="symm")
    A_Jaccard = A_Jaccard >= 1


    AInterB = VT_ImageNB_grossie * A_Jaccard
    AUnionB = (VT_ImageNB_grossie + A_Jaccard) >= 1
    AprivéB = (VT_ImageNB_grossie * (~(AInterB))) >= 1
    BprivéA = (A_Jaccard * (~(AInterB))) >= 1

    # plt.figure()
    # plt.imshow(AInterB > 0)
    # plt.figure()
    # plt.imshow(AUnionB > 0)
    # plt.show()

    indice_Jaccard = np.sum(AInterB > 0) / np.sum(AUnionB > 0)

    Tversky = np.sum(AInterB > 0) / (np.sum(AInterB > 0) + α * np.sum(AprivéB) + β * np.sum(BprivéA))

    print("Résultat U-Net : Indice de Jaccard :",indice_Jaccard)
    print("Résultat U-Net : Indice de Tversky :",Tversky)

    a, b = ot.unif(ns), ot.unif(nt)  # uniform distribution on samples
    print("ns,nt=",ns,nt)

    # loss matrix
    M = ot.dist(Index_VT, Index_A, metric='sqeuclidean').astype('float64')
    # M /= M.max()

    Wasserstein = np.sqrt(ot.emd2(a, b, M))

    print("Résultat U-Net : Wasserstein :", Wasserstein)

    A = np.zeros((n,n))

    for (nœud1,nœud2),filtr in Filtration_Arêtes_Arbre :
        i1,j1 = nœud1//n, nœud1%n
        x1,y1 = j1, i1
        i2,j2 = nœud2//n, nœud2%n
        x2,y2 = j2, i2
        dessine_segment(A,x1,y1,x2,y2)
        # plt.plot([x1,x2],[y1,y2],c='r',alpha=0.5)
        # if nœud1 in Centres and not nœud1 in Centres_dessinés :
        #     plt.scatter(x1,y1, c='k', marker='*')
        #     Centres_dessinés.add(nœud1)
        # elif nœud2 in Centres and not nœud2 in Centres_dessinés :
        #     plt.scatter(x2,y2, c='k', marker='*')
        #     Centres_dessinés.add(nœud1)


        if filtr < rayon_suivant :

            print("Nouvelle distance de Wasserstein à calculer")
            print("Filtr = ", filtr)



            Index_A = np.argwhere(A.T >= 1)
            nt,_ = Index_A.shape

            A_bin = A.T >= 1
            A_Jaccard = convolve2d(A_bin, Kernel, mode="same", boundary="symm")
            A_Jaccard = A_Jaccard >= 1

            if filtr < 0.3 and not déjà_fait :
                plot_fault_overlays(Image[δn:-δn,δn:-δn], VT_ImageNB[δn:-δn,δn:-δn], A.T, résultats_UNet)#, title="Réseaux de failles : GT vs A vs U-Net")#, savepath="overlay_faults.png")
                déjà_fait = True
                plt.figure()
                plt.imshow(A.T >= 1)
                plt.colorbar()
                plt.figure()
                plt.imshow(VT_ImageNB)
                plt.colorbar()
                plt.figure()
                plt.imshow(VT_ImageNB_grossie.astype(float) + A_Jaccard.astype(float))
                plt.colorbar()
                plt.show()


            AInterB = VT_ImageNB_grossie * A_Jaccard
            AUnionB = (VT_ImageNB_grossie + A_Jaccard) >= 1
            AprivéB = (VT_ImageNB_grossie * (~(AInterB))) >= 1
            BprivéA = (A_Jaccard * (~(AInterB))) >= 1

            # plt.figure()
            # plt.imshow(AInterB > 0)
            # plt.figure()
            # plt.imshow(AUnionB > 0)
            # plt.show()

            indice_Jaccard = np.sum(AInterB > 0) / np.sum(AUnionB > 0)

            Tversky = np.sum(AInterB > 0) / (np.sum(AInterB > 0) + α * np.sum(AprivéB) + β * np.sum(BprivéA))

            print("Indice de Jaccard :",indice_Jaccard)
            print("Indice de Tversky :",Tversky)
            if indice_Jaccard > Meilleures_filtrations["Jaccard"][0] :
                Meilleures_filtrations["Jaccard"][0], Meilleures_filtrations["Jaccard"][1] = indice_Jaccard, filtr

            if Tversky > Meilleures_filtrations["Tversky"][0] :
                Meilleures_filtrations["Tversky"][0], Meilleures_filtrations["Tversky"][1] = Tversky, filtr


            a, b = ot.unif(ns), ot.unif(nt)  # uniform distribution on samples
            print("ns,nt=",ns,nt)

            # loss matrix
            M = ot.dist(Index_VT, Index_A, metric='sqeuclidean').astype('float64')
            # M /= M.max()

            Wasserstein = np.sqrt(ot.emd2(a, b, M))

            if Wasserstein < Meilleures_filtrations["Wasserstein"][0] :
                Meilleures_filtrations["Wasserstein"][0], Meilleures_filtrations["Wasserstein"][1] = Wasserstein, filtr

            print("Distance de Wasserstein : ", Wasserstein)

            index_rayon += 1
            if index_rayon >= len(Rayons) :
                break
            else :
                rayon_suivant = Rayons[index_rayon]

    print(Meilleures_filtrations)